# Практическая работа № 4 — Регрессионная модель (уровень B)

**Студент:** Barsbek Konurbaev

Цель: сравнить линейную регрессию на полном и сокращённом наборе признаков, оценить мультиколлинеарность и изучить остатки.

## 1. Загрузка реального датасета
Используется Diabetes dataset из scikit-learn: реальные медицинские измерения и числовая целевая переменная прогрессирования диабета. Это регрессия, данные не синтетические.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = load_diabetes(as_frame=True)
X = data.data
y = data.target
print("Размер:", X.shape)
display(X.head())
display(X.describe().T)

## 2. Train/test и базовая линейная регрессия

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
baseline = LinearRegression()
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

def regression_metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred)
    }

baseline_metrics = regression_metrics(y_test, pred_baseline)
display(pd.DataFrame([baseline_metrics], index=["Все признаки"]).round(4))

## 3. Проверка мультиколлинеарности

In [ ]:
corr = X_train.corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Корреляция")
plt.xticks(range(len(X.columns)), X.columns, rotation=90)
plt.yticks(range(len(X.columns)), X.columns)
plt.title("Корреляционная матрица признаков")
plt.tight_layout()
plt.show()

pairs = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
display(pairs.sort_values(ascending=False).head(10).to_frame("absolute_correlation"))

## 4. Модель на сокращённом наборе признаков
Удаляем один признак из каждой пары с абсолютной корреляцией выше 0.85, затем сравниваем результат с базовой моделью.

In [ ]:
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_cols = [col for col in upper.columns if (upper[col] > 0.85).any()]
selected_features = [col for col in X.columns if col not in drop_cols]
print("Удалённые признаки:", drop_cols)
print("Оставшиеся признаки:", selected_features)

reduced = LinearRegression()
reduced.fit(X_train[selected_features], y_train)
pred_reduced = reduced.predict(X_test[selected_features])

comparison = pd.DataFrame(
    [baseline_metrics, regression_metrics(y_test, pred_reduced)],
    index=["Все признаки", "Сокращённые признаки"]
)
display(comparison.round(4))

## 5. Анализ остатков

In [ ]:
residuals = y_test - pred_baseline
plt.figure(figsize=(7, 4))
plt.scatter(pred_baseline, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Остатки базовой линейной регрессии")
plt.xlabel("Предсказанное значение")
plt.ylabel("Остаток (истинное − предсказанное)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_baseline, alpha=0.7)
lims = [min(y_test.min(), pred_baseline.min()), max(y_test.max(), pred_baseline.max())]
plt.plot(lims, lims, linestyle="--")
plt.title("Истинные значения против предсказаний")
plt.xlabel("Истинное значение")
plt.ylabel("Предсказание")
plt.tight_layout()
plt.show()

## 6. Вывод
Сравните MAE, RMSE и R² обеих моделей. MAE показывает среднюю абсолютную ошибку в единицах целевой переменной. Решение об исключении признаков принимайте по тестовым метрикам и графику остатков, а не только по корреляции.